# Linear Regression

This notebook is a comprehensive guide to **Linear Regression**, following the official [Google Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course/linear-regression).

We will use the real-world problem of **predicting taxi fares** (which is the total cost a passenger pays for a vehicle trip, typically consisting of a base fee plus a rate based on distance or time) to explain every core concept of Machine Learning, from simple lines to multiple features.

---

## 🚕 1. The Core Concepts: What is Linear Regression?

**Linear Regression** is a method to find the best relationship between a **Label** (what we want to predict) and one or more **Features** (what we know).

### Simple Example (Single Feature):
To predict the fare based only on distance:
- **Label (y)**: The `FARE` ($)
- **Feature (x)**: `TRIP_MILES`
- **Formula**: `y = wx + b`

### Real-World Example (Multiple Features):
In the real world, fares depend on more than just distance. For example, time spent in traffic also increases the cost.
- **Label (y)**: The `FARE` ($)
- **Feature 1 (x₁)**: `TRIP_MILES` (distance)
- **Feature 2 (x₂)**: `TRIP_MINUTES` (time)
- **Formula**: `y = w₁x₁ + w₂x₂ + b`

---

### The Mathematical Model
We represent this relationship as a straight line equation:
**$$y = wx + b$$**

1. **$y'$ (y-prime)**: The predicted value (the Fare we guess).
2. **$w$ (Weight)**: The slope. In our case, this is the **Price per Mile**.
3. **$x$**: The input. This is the **Distance (Miles)**.
4. **$b$ (Bias)**: The y-intercept. This is the **Base Fee** (the cost just for getting in the taxi).

**Training** a model simply means finding the "perfect" values for $w$ and $b$ that best fit our historical taxi data.

## 📉 2. Measuring Success: The Loss Function

In Machine Learning, we need a mathematical way to tell the model, "You are wrong by this much." We call this **Loss**.

As explained in [Types of Loss](https://developers.google.com/machine-learning/crash-course/linear-regression/loss#types_of_loss), here are the five main types of loss used in Linear Regression:

| Loss Type | Definition | Key Characteristic |
| :--- | :--- | :--- |
| **L1 Loss** | The absolute difference between the predicted and actual value for a **single** example. | $Loss = |y - y'|$ |
| **Mean Absolute Error (MAE)** | The **average** of L1 losses across a set of $N$ examples. | Provides a linear score of error. |
| **L2 Loss** | The squared difference between the predicted and actual value for a **single** example. | $Loss = (y - y')^2$ |
| **Mean Squared Error (MSE)** | The **average** of L2 losses across a set of $N$ examples. | Punishes large outliers heavily. |
| **Root Mean Squared Error (RMSE)** | The **square root** of the Mean Squared Error (MSE). | Brings the error unit back to the original scale (e.g., Dollars). |

---

### ⚖️ Choosing a Loss Function (MSE vs. MAE)

According to the [ML Crash Course](https://developers.google.com/machine-learning/crash-course/linear-regression/loss#choosing_a_loss), your choice depends on how you want to handle **outliers** (data points that are very different from the rest):

*   **Choose MSE (L2)** if you want your model to stay very far away from large errors. Because errors are squared, a single outlier can significantly increase the total loss, forcing the model to adjust more to accommodate it.
*   **Choose MAE (L1)** if you want your model to be more **robust** to outliers. In MAE, a point that is $100$ miles off is only $10$ times worse than a point $10$ miles off. In MSE, it would be $100$ times worse.

### Why use L2/MSE most of the time?
1. **Removes negative signs**: An over-prediction doesn't cancel out an under-prediction.
2. **Mathematical efficiency**: MSE is easier to calculate gradients for (it's a smooth curve), which helps the model learn faster during optimization.

**The MSE Formula:**
$$MSE = \frac{1}{N} \sum_{(x,y) \in D} (y - (wx + b))^2$$

#TODO Numpy

In [1]:
import numpy as np

# Let's say we have 3 taxi trips
actual_fares = np.array([12.00, 25.00, 40.00])
predicted_fares = np.array([13.50, 21.00, 45.00])  # Our model's guesses

# 1. L1 Loss (Absolute Error for each trip)
l1_losses = np.abs(actual_fares - predicted_fares)

# 2. MAE (Mean Absolute Error)
mae = np.mean(l1_losses)

# 3. L2 Loss (Squared Error for each trip)
l2_losses = np.square(actual_fares - predicted_fares)

# 4. MSE (Mean Squared Error)
mse = np.mean(l2_losses)

# 5. RMSE (Root Mean Squared Error)
rmse = np.sqrt(mse)

print(f"Actual Fares:    {actual_fares}")
print(f"Predicted Fares: {predicted_fares}")
print("-" * 30)
print(f"L1 Losses (per trip):  {l1_losses}")
print(f"MAE (Average L1):      ${mae:.2f}")
print(f"L2 Losses (per trip):  {l2_losses}")
print(f"MSE (Average L2):      {mse:.2f}")
print(f"RMSE (Standard Error): ${rmse:.2f}")

Actual Fares:    [12. 25. 40.]
Predicted Fares: [13.5 21.  45. ]
------------------------------
L1 Losses (per trip):  [1.5 4.  5. ]
MAE (Average L1):      $3.50
L2 Losses (per trip):  [ 2.25 16.   25.  ]
MSE (Average L2):      14.42
RMSE (Standard Error): $3.80


## 🔄 3. Optimization: Gradient Descent

How does the model actually "learn"? It uses an iterative approach called **Gradient Descent** to reduce the loss.

### The "Foggy Mountain" Analogy
Imagine you are at the top of a mountain (High Loss) in thick fog. You want to reach the bottom (Low Loss).
1. **Calculate Gradient**: You feel the slope with your feet to see which way is "down."
2. **Step Down**: You take a small step in that direction.
3. **Repeat**: You keep doing this until the ground is flat (Convergence).

### Deriving the Gradient (The "Fix")
To know the slope, we take the derivative of the MSE formula with respect to the weight ($w$):
1. Start with $Loss = (y - (wx + b))^2$
2. Use the **Chain Rule**: The derivative of $(\text{something})^2$ is $2 \times (\text{something}) \times (\text{derivative of inner part})$.
3. The derivative of $(y - wx - b)$ with respect to $w$ is $-x$.
4. **The Derived Gradient**: $Gradient = -2x(y - y_{predicted}) = -2x(Error)$.

### The Update Rule
**$$w_{new} = w_{old} - (Learning\_Rate \times Gradient)$$**

Let's see this calculation in code!

### ⛰️ Convergence and Convex Functions

For more detail, see the [ML Crash Course: Convergence and Convex Functions](https://developers.google.com/machine-learning/crash-course/linear-regression/gradient-descent#convergence_and_convex_functions).

When we train a model, we are looking for the point where the loss is as low as possible. Two concepts make this possible for Linear Regression:

#### 1. Convex Functions (The "Bowl")
In Linear Regression, the Mean Squared Error (MSE) loss function is **convex**. This means it is shaped like a bowl.
*   **One Bottom**: A convex function has only one minimum point (the **Global Minimum**).
*   **No Traps**: There are no "dips" or local pits where the model could get stuck. No matter where you start on the bowl, going "downhill" will always lead you to the same bottom.

#### 2. Convergence
**Convergence** is the state where the model has finished training.
*   As the model gets closer to the bottom of the bowl, the gradient (the slope) becomes flatter and flatter.
*   Eventually, the updates to the weights become so tiny that the loss stops changing.
*   At this point, we say the model has **converged** on the optimal solution.

In [ ]:
# Manual Gradient Descent: Reaching Convergence
weight = 10.0  # Initial guess
learning_rate = 0.05
actual_fare = 25.0
miles = 2.0

# Convergence settings
threshold = 0.001  # Stop when weight changes by less than this
max_iterations = 100

print(f"Target Fare: ${actual_fare:.2f} | Starting Weight: {weight}\n")
print(f"{'Iter':<5} | {'Weight':<10} | {'Prediction':<12} | {'Error':<10}")
print("-" * 50)

for i in range(max_iterations):
    # 1. Prediction and Error
    prediction = weight * miles
    error = prediction - actual_fare

    # 2. Calculate Gradient
    gradient = miles * error

    # 3. Update Weight
    adjustment = learning_rate * gradient
    new_weight = weight - adjustment

    # Print progress every few steps or at the end
    if i < 5 or abs(adjustment) < threshold:
        print(f"{i+1:<5} | {weight:<10.4f} | ${prediction:<11.2f} | {error:<10.2f}")

    # 4. Check for Convergence
    if abs(adjustment) < threshold:
        print("-" * 50)
        print(f"✅ Converged at iteration {i+1}!")
        print(f"Final Learned Weight: {weight:.4f}")
        print(f"Final Prediction:     ${prediction:.2f}")
        break

    weight = new_weight

## ⚙️ 4. Hyperparameters: The Control Knobs

Hyperparameters are settings we choose manually to control the learning process.

- 🎓 **Learning Rate (Step Size)**:
    - Too high: The model "overshoots" the valley and jumps around (Unstable).
    - Too low: The model takes too many tiny steps (Slow).
- 🔄 **Batch Size**: The number of examples the model looks at before updating the weights. Looking at *all* data is slow, so we use small "batches."
- 🔄 **Epochs**: One complete pass through the entire dataset.
- 🏁 **Convergence**: When the Loss stops decreasing significantly, the model has finished learning.

## 🚀 5. Scaling Up: Multiple Features

Real-world problems are rarely solved by just one feature. For our taxi example, **Distance** (Miles) is important, but **Time** (Minutes) also matters because drivers charge for time spent in traffic.

### Multiple Linear Regression Formula
We simply add more weights for each new feature:
**$$y = w_1x_1 + w_2x_2 + b$$**

- **$w_1$**: Weight for Miles (Price per mile).
- **$x_1$**: Trip Miles.
- **$w_2$**: Weight for Time (Price per minute).
- **$x_2$**: Trip Minutes.

**The Improvement**: By adding more information, our model can make much more accurate predictions. For example, a 1-mile trip in heavy traffic will now correctly cost more than a 1-mile trip on an empty road.

---
## 🛠️ 6. Real-World Implementation with TensorFlow

Now we will apply these concepts to thousands of Chicago Taxi trips.

In [ ]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Load Data
DATA_URL = 'https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv'
df = pd.read_csv(DATA_URL)
training_df = df.loc[:, ('TRIP_MILES', 'TRIP_SECONDS', 'FARE')].dropna()

# Feature Engineering: Convert seconds to minutes
training_df['TRIP_MINUTES'] = training_df['TRIP_SECONDS'] / 60.0

print(f"Loaded {len(training_df):,} trips.")

### Defining our Automated Solver

**Complex Functions Explained:**
- `tf.keras.layers.Dense(units=1)`: Creates our mathematical line. If we give it 1 feature, it's $y=wx+b$. If we give it 2 features, it automatically becomes $y=w_1x_1+w_2x_2+b$.
- `optimizer=RMSprop`: Our Gradient Descent Engine. For multiple features, it calculates the gradient for **every weight** ($w_1, w_2$) and updates them simultaneously.

In [ ]:
def build_model(learning_rate, feature_count):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=1, input_shape=(feature_count,))
    ])
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )
    return model

def train_model(model, df, feature_list, label, epochs, batch_size):
    history = model.fit(x=df[feature_list], y=df[label], batch_size=batch_size, epochs=epochs, verbose=0)
    return history

## 🧪 7. Experiments: The Power of Multiple Features

Let's compare the **Single Feature** model (Miles only) against the **Multiple Feature** model (Miles + Minutes) to see the improvement.

In [ ]:
learning_rate = 0.01
epochs = 30
batch_size = 50

# 1. Train Model with 1 Feature (Miles)
model_1 = build_model(learning_rate, feature_count=1)
history_1 = train_model(model_1, training_df, ['TRIP_MILES'], 'FARE', epochs, batch_size)

# 2. Train Model with 2 Features (Miles + Minutes)
model_2 = build_model(learning_rate, feature_count=2)
history_2 = train_model(model_2, training_df, ['TRIP_MILES', 'TRIP_MINUTES'], 'FARE', epochs, batch_size)

plt.figure(figsize=(10, 6))
plt.plot(history_1.history['root_mean_squared_error'], label='1 Feature (Miles Only)', color='orange')
plt.plot(history_2.history['root_mean_squared_error'], label='2 Features (Miles + Minutes)', color='teal')
plt.title('Improvement: 1 Feature vs. Multiple Features')
plt.xlabel('Epochs')
plt.ylabel('Error ($)')
plt.legend()
plt.grid(True)
plt.show()

final_err_1 = history_1.history['root_mean_squared_error'][-1]
final_err_2 = history_2.history['root_mean_squared_error'][-1]
improvement = ((final_err_1 - final_err_2) / final_err_1) * 100

print(f"Final Error (Miles Only):    ${final_err_1:.2f}")
print(f"Final Error (Miles + Mins):  ${final_err_2:.2f}")
print(f"Total Accuracy Improvement:  {improvement:.1f}%")

## 🎯 8. Final Accuracy Check
Let's see our best model (Multiple Features) in action!

In [ ]:
sample_df = training_df.sample(500)
predictions = model_2.predict(sample_df[['TRIP_MILES', 'TRIP_MINUTES']])

plt.figure(figsize=(10, 6))
plt.scatter(sample_df['FARE'], predictions, alpha=0.5, color='teal')
plt.plot([0, 80], [0, 80], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Fare ($)')
plt.ylabel('Predicted Fare ($)')
plt.title('Final Model Accuracy (Miles + Minutes)')
plt.legend()
plt.show()